# Prepare matrices for SCENICplus

Plan: Run SCENIC+ on subsampled dataset to max 2k cells per celltype

In [1]:
###################
## Load packages
###################
suppressPackageStartupMessages({
    library(scran)
    library(scater)
    library(Seurat)
    library(ArchR)
})


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .______      
          /   \     |   _ 

In [2]:
###################
## I/O
###################
io = list()
io$basedir = '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/'

## Single-cell
io$RNA_sce = file.path(io$basedir,"data/processed/rna/SingleCellExperiment.rds")
io$meta = file.path(io$basedir,"results/atac/archR/qc/sample_metadata_after_qc.txt.gz")

# ArchR vitro
io$archrvitro = file.path("/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/processed/atac/archR")
# ArchR vivo
io$archrvivo = file.path(io$basedir,"data/processed/atac/archR")

# output
io$outdir = file.path("/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/results/multiome_atlas/SCENICplus_sc/")
dir.create(io$outdir, recursive=TRUE, showWarnings =FALSE)

In [3]:
meta = fread(io$meta)[pass_rnaQC == T & pass_atacQC == T & doublet_call == F & !sample %in% c('E8.5_CRISPR_T_KO', 'E8.5_CRISPR_T_WT')]
nrow(meta)

[1] 45713

In [4]:
archrvivo = loadArchRProject(io$archrvivo)[meta$cell]
archrvitro = loadArchRProject(io$archrvitro)

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [5]:
archrvivo


           ___      .______        ______  __    __  .______      
          /   \     |   _  \      /      ||  |  |  | |   _  \     
         /  ^  \    |  |_)  |    |  ,----'|  |__|  | |  |_)  |    
        /  /_\  \   |      /     |  |     |   __   | |      /     
       /  _____  \  |  |\  \\___ |  `----.|  |  |  | |  |\  \\___.
      /__/     \__\ | _| `._____| \______||__|  |__| | _| `._____|
    



class: ArchRProject 
outputDirectory: /rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/data/processed/atac/archR 
samples(11): E7.5_rep1 E7.5_rep2 ... E8.5_CRISPR_T_KO E8.5_CRISPR_T_WT
sampleColData names(1): ArrowFiles
cellColData names(35): Sample TSSEnrichment ... ReadsInPeaks FRIP
numberOfCells(1): 45713
medianTSS(1): 16.355
medianFrags(1): 30489

In [6]:
# Add in vitro peak matrix to vivo ArchR object
vitroPeaks = getPeakSet(archrvitro)

In [7]:
addArchRThreads(24)
archrvivo = addFeatureMatrix(
              input = archrvivo,
              features = vitroPeaks,
              matrixName = "vitroPeakMatrix",
              ceiling = 4,
              binarize = FALSE,
              verbose = TRUE,
              threads = getArchRThreads(),
              parallelParam = NULL,
              force = TRUE,
              logFile = createLogFile("vitroPeakMatrix")
            )

Setting default number of Parallel threads to 24.

ArchR logging to : ArchRLogs/ArchR-vitroPeakMatrix-f8945749f5ae-Date-2024-10-30_Time-15-06-40.log
If there is an issue, please report to github with logFile!

2024-10-30 15:06:44 : Batch Execution w/ safelapply!, 0 mins elapsed.

ArchR logging successful to : ArchRLogs/ArchR-vitroPeakMatrix-f8945749f5ae-Date-2024-10-30_Time-15-06-40.log



In [8]:
addArchRThreads(5)
atac.sce = getMatrixFromProject(archrvivo, useMatrix = 'vitroPeakMatrix')

Setting default number of Parallel threads to 5.

ArchR logging to : ArchRLogs/ArchR-getMatrixFromProject-f8942380411-Date-2024-10-30_Time-15-10-39.log
If there is an issue, please report to github with logFile!

2024-10-30 15:11:49 : Organizing colData, 1.161 mins elapsed.

2024-10-30 15:11:50 : Organizing rowData, 1.169 mins elapsed.

2024-10-30 15:11:50 : Organizing rowRanges, 1.169 mins elapsed.

2024-10-30 15:11:50 : Organizing Assays (1 of 1), 1.169 mins elapsed.

2024-10-30 15:12:25 : Constructing SummarizedExperiment, 1.764 mins elapsed.

2024-10-30 15:13:10 : Finished Matrix Creation, 2.511 mins elapsed.



In [9]:
######
## subset to shared cells
######

# Load RNA SingleCellExperiment
rna.sce <- readRDS(io$RNA_sce)
#rna.sce = rna.sce[,match(meta$cell, colnames(rna.sce))]
# Make sure that samples are consistent
cells <- intersect(colnames(rna.sce),colnames(atac.sce))
rna.sce <- rna.sce[,cells]
atac.sce <- atac.sce[,cells]

In [18]:
######
## subset to X cells per celltype
######
meta = meta[match(cells, cell)]
n_subsample = 1000

celltype_subsampling = mclapply(unique(meta$celltype), function(x){
    tmp = meta[celltype == x]
    if(nrow(tmp)<n_subsample){
        cells = tmp$cell
    }else{
        cells = sample(tmp$cell, n_subsample)
    }
    return(cells)
})

celltype_subsampling = unlist(celltype_subsampling)

In [20]:
head(celltype_subsampling)

[1] "E7.5_rep1#AAACAGCCATCCTGAA-1" "E7.5_rep1#AACGCTAGTTGTAAAC-1"
[3] "E7.5_rep1#AAGCCTGTCATAAGCC-1" "E7.5_rep1#AAGGTGCAGAGGAAGG-1"
[5] "E7.5_rep1#AAGTTTGTCCCATAGG-1" "E7.5_rep1#AATCTCAAGCTCAAAC-1"

In [21]:
meta_subsampled = meta[match(celltype_subsampling, cell)]

In [25]:
rna.sce <- rna.sce[,meta_subsampled$cell]
atac.sce <- atac.sce[,meta_subsampled$cell]

In [26]:
rna.sce
atac.sce

class: SingleCellExperiment 
dim: 32285 25665 
metadata(0):
assays(1): counts
rownames(32285): Xkr4 Gm1992 ... AC234645.1 AC149090.1
rowData names(0):
colnames(25665): E7.5_rep1#AAACAGCCATCCTGAA-1
  E7.5_rep1#AACGCTAGTTGTAAAC-1 ... E8.75_rep2#TTTGAGTCAAGGTAAC-1
  E8.75_rep2#TTTGCGGAGTAAGAAC-1
colData names(10): barcode sample ... pass_rnaQC sizeFactor
reducedDimNames(0):
mainExpName: RNA
altExpNames(0):

class: RangedSummarizedExperiment 
dim: 234908 25665 
metadata(0):
assays(1): vitroPeakMatrix
rownames: NULL
rowData names(1): idx
colnames(25665): E7.5_rep1#AAACAGCCATCCTGAA-1
  E7.5_rep1#AACGCTAGTTGTAAAC-1 ... E8.75_rep2#TTTGAGTCAAGGTAAC-1
  E8.75_rep2#TTTGCGGAGTAAGAAC-1
colData names(35): BlacklistRatio nDiFrags ... ReadsInPeaks FRIP

In [27]:
atac.sce = as(atac.sce, 'SingleCellExperiment')

In [28]:
# Add peak name to atac.pb

In [30]:
rownames(atac.sce) = paste0(seqnames(vitroPeaks), ':', start(vitroPeaks), '-', end(vitroPeaks))

In [31]:
bed = data.frame(chr = seqnames(vitroPeaks),
                 start = start(vitroPeaks), 
                 end = end(vitroPeaks))

In [32]:
fwrite(bed, col.names = F, file.path(io$outdir, 'regions.bed'), sep = '\t')

In [33]:
# Save metadata
fwrite(meta_subsampled, file.path(io$outdir, 'meta_subsampled.txt'))

In [34]:
## Save output for SCENIC+
write.table(colnames(rna.sce), file.path(io$outdir, 'cells.txt'))
write.table(rownames(rna.sce), file.path(io$outdir, 'genes.txt'))
write.table(rownames(atac.sce), file.path(io$outdir, 'peaks.txt'), sep = '\t', row.names = F)

# Save RNA as mtx file
writeMM(as(counts(rna.sce), "dgCMatrix"), file=file.path(io$outdir, 'rna.mtx'))
assayNames(atac.sce) = 'counts'
writeMM(as(counts(atac.sce), "dgCMatrix"), file=file.path(io$outdir, 'atac.mtx'))


NULL

NULL